# Notebook 01 — La escalera a mano

Este notebook acompaña la sección 21.2. Primero vas a llenar la tabla de valores **a mano** en la celda siguiente, y después vas a verificar tus cálculos con código.

## Problema

- Estados: $0, 1, 2, 3, 4, 5$
- Costos por escalón: $c = [3, 2, 5, 10, 1, 0]$
- Acciones: $\{\text{subir 1}, \text{saltar 2}\}$
- Meta: llegar al escalón 5 con costo total mínimo.
- Convención: $V(i)$ es el costo *futuro* desde $i$ (no incluye $c_i$). El costo se paga al *aterrizar* en un escalón.


## Paso 1 — Llena la tabla *antes* de correr nada

Recorre la tabla de derecha a izquierda. Para cada $i$:

$$V(i) = c_i + \min_a V(T(i, a))$$

| $i$ | $c_i$ | $V(i)$ |
|:---:|:-----:|:------:|
| 5 | 0 | _tu respuesta_ |
| 4 | 1 | _tu respuesta_ |
| 3 | 10 | _tu respuesta_ |
| 2 | 5 | _tu respuesta_ |
| 1 | 2 | _tu respuesta_ |
| 0 | 3 | _tu respuesta_ |

Cuando termines, ejecuta la siguiente celda para verificar.


In [ ]:
import numpy as np

COSTS = np.array([3, 2, 5, 10, 1, 0])
N_STATES = len(COSTS)
GOAL = N_STATES - 1


def bellman_deterministic(costs):
    """Tabulación bottom-up: V(i) = min_a { c(i,a) + V(T(i,a)) }.

    V(i) NO incluye c_i (costo del estado actual); cuenta solo los costos
    futuros que aún tienes que pagar desde i hasta llegar a la meta.
    """
    n = len(costs)
    V = np.zeros(n)
    # V[n-1] = 0 ya (meta, no falta pagar nada)
    for i in range(n - 2, -1, -1):
        # acción "subir 1": pagas c[i+1] al aterrizar, luego V[i+1]
        options = [costs[i + 1] + V[i + 1]]
        if i + 2 < n:
            # acción "saltar 2": pagas c[i+2] al aterrizar, luego V[i+2]
            options.append(costs[i + 2] + V[i + 2])
        V[i] = min(options)
    return V


V_det = bellman_deterministic(COSTS)
print("V =", V_det.tolist())

# Verificación contra los valores esperados
expected = [6, 6, 1, 0, 0, 0]
assert np.allclose(V_det, expected), f"Se esperaba {expected}, se obtuvo {V_det.tolist()}"
print("✓ La tabla coincide con lo calculado a mano.")


## Paso 2 — Extraer la política óptima

La función de valor te dice *qué tan bueno* es cada estado. La política $\pi^*$ te dice *qué hacer*. La extraemos guardando el $\arg\min$ junto con el $\min$.


In [ ]:
def extract_policy_and_trajectory(costs, V):
    """Devuelve (política, trayectoria desde 0, costo total).

    El costo total es la suma de costos de los escalones donde ATERRIZAS
    (no incluye el escalón inicial 0, porque no aterrizaste ahí).
    """
    n = len(costs)
    policy = [None] * n
    for i in range(n - 1):
        # Evaluar cada acción: costo = c[destino] + V[destino]
        options = [("subir 1", i + 1, costs[i + 1] + V[i + 1])]
        if i + 2 < n:
            options.append(("saltar 2", i + 2, costs[i + 2] + V[i + 2]))
        policy[i] = min(options, key=lambda t: t[2])
    # Construir trayectoria, sumando c[s'] en cada transición
    traj = [0]
    i = 0
    total = 0
    while i < n - 1:
        action, next_i, _ = policy[i]
        total += costs[next_i]   # pagamos el costo del destino al aterrizar
        i = next_i
        traj.append(i)
    return policy, traj, total


policy, traj, total_cost = extract_policy_and_trajectory(COSTS, V_det)
for i, p in enumerate(policy):
    if p:
        action, next_i, val = p
        print(f"  estado {i}: {action} (→ {next_i}, valor acumulado {val})")
print(f"\nTrayectoria: {traj}")
print(f"Costo total:  {total_cost}")

assert traj == [0, 2, 4, 5], f"Se esperaba [0, 2, 4, 5], se obtuvo {traj}"
assert total_cost == 6
print("✓ La política óptima recorre 0 → 2 → 4 → 5 con costo 6.")


## Paso 3 — Extensión estocástica

Ahora la acción *saltar 2* tiene probabilidad 0.8 de éxito y 0.2 de resbalar (solo avanzas 1). La ecuación se vuelve:

$$V(s) = \min_a \sum_{s'} P(s' \mid s, a)\bigl[ c(s, a, s') + V(s') \bigr]$$


In [ ]:
P_SUCCESS = 0.8
P_SLIP = 0.2


def bellman_stochastic(costs, p_success=P_SUCCESS, p_slip=P_SLIP):
    """Bellman estocástico. Costo pagado al aterrizar en el destino.

    Acción "subir 1": determinista → i+1, pagas c[i+1].
    Acción "saltar 2": lands at i+2 w.p. p_success, at i+1 w.p. p_slip;
                       el costo esperado es el costo del destino esperado.
    """
    n = len(costs)
    V = np.zeros(n)
    for i in range(n - 2, -1, -1):
        # Acción 'subir 1': pagas c[i+1], luego V[i+1]
        val_up1 = costs[i + 1] + V[i + 1]
        options = [val_up1]
        # Acción 'saltar 2' (si i+2 existe):
        #   E[c + V(next)] = p_success * (c[i+2] + V[i+2]) + p_slip * (c[i+1] + V[i+1])
        if i + 2 < n:
            val_up2 = (p_success * (costs[i + 2] + V[i + 2])
                       + p_slip   * (costs[i + 1] + V[i + 1]))
            options.append(val_up2)
        V[i] = min(options)
    return V


V_stoch = bellman_stochastic(COSTS)
print("V estocástico =", V_stoch.tolist())

expected_stoch = [8.24, 7.84, 2.84, 0.2, 0.0, 0.0]
assert np.allclose(V_stoch, expected_stoch, atol=1e-9), (
    f"Se esperaba {expected_stoch}, se obtuvo {V_stoch.tolist()}"
)

print("\nComparación:")
print(f"  Determinista: V(0) = {V_det[0]}")
print(f"  Estocástico:  V(0) = {V_stoch[0]}")
print(f"  Precio de la incertidumbre: {V_stoch[0] - V_det[0]:.2f}")


## Paso 4 — Con descuento $\gamma$

Si además valoramos menos el futuro, multiplicamos la esperanza por $\gamma$:

$$V(s) = \min_a \sum_{s'} P(s' \mid s, a)\bigl[ c(s, a, s') + \gamma V(s') \bigr]$$


In [ ]:
def bellman_discounted(costs, gamma=0.95, p_success=P_SUCCESS, p_slip=P_SLIP):
    """Bellman estocástico con descuento gamma. γ descuenta el VALOR FUTURO,
    no el costo inmediato (que se paga ahora)."""
    n = len(costs)
    V = np.zeros(n)
    for i in range(n - 2, -1, -1):
        # subir 1: c[i+1] + γ·V[i+1]
        val_up1 = costs[i + 1] + gamma * V[i + 1]
        options = [val_up1]
        if i + 2 < n:
            # saltar 2: E_{s'}[c[s'] + γ·V[s']]
            val_up2 = (p_success * (costs[i + 2] + gamma * V[i + 2])
                       + p_slip   * (costs[i + 1] + gamma * V[i + 1]))
            options.append(val_up2)
        V[i] = min(options)
    return V


for g in [1.0, 0.95, 0.9, 0.5]:
    V_g = bellman_discounted(COSTS, gamma=g)
    vals = [round(float(v), 3) for v in V_g]
    print(f"γ = {g:4.2f}: V = {vals}")


## Paso 5 — Visualizar la trayectoria óptima


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 4))
step_w, step_h = 1.2, 0.5
for i in range(N_STATES):
    x = i * step_w
    y = i * step_h
    color = "#2E86AB" if i in traj else "#ECF0F1"
    rect = mpatches.Rectangle((x, y), step_w, step_h, facecolor=color,
                               edgecolor="#2C3E50", linewidth=1.2, alpha=0.7 if i in traj else 1.0)
    ax.add_patch(rect)
    ax.text(x + step_w / 2, y + step_h / 2 + 0.05, f"i={i}", ha="center", va="center",
            fontsize=11, fontweight="bold")
    ax.text(x + step_w / 2, y + step_h / 2 - 0.13, f"V={V_det[i]:.0f}", ha="center", va="center",
            fontsize=9, color="#2E86AB")

# Flechas de la trayectoria
for a, b in zip(traj[:-1], traj[1:]):
    ax.annotate("", xy=(b * step_w + step_w / 2, b * step_h + step_h + 0.08),
                xytext=(a * step_w + step_w / 2, a * step_h + step_h + 0.08),
                arrowprops=dict(arrowstyle="->", color="#27AE60", lw=2))

ax.set_xlim(-0.3, N_STATES * step_w + 0.5)
ax.set_ylim(-0.3, N_STATES * step_h + 0.8)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(f"Trayectoria óptima (determinista): {traj}, costo = {total_cost}",
             fontsize=12)
plt.tight_layout()
plt.show()


## Resumen

En este notebook:

1. **Calculaste a mano** los seis valores de la tabla determinista (debieron dar $V = [6, 6, 1, 0, 0, 0]$).
2. **Verificaste tu cálculo con código** — usando tabulación bottom-up.
3. **Extrajiste la política óptima** $\pi^*$ y la trayectoria $0 \to 2 \to 4 \to 5$.
4. **Añadiste incertidumbre** (probabilidad de resbalar) y viste el *precio de la incertidumbre*: $V(0)$ pasó de 6 a 8.24.
5. **Incorporaste descuento** $\gamma$ y observaste cómo el futuro se vuelve más ligero conforme $\gamma$ baja.

En el siguiente notebook vamos a **medir** la diferencia entre recursión ingenua y programación dinámica.
